# Mumbai monthly rental regression — Day 2

The submitted model is a Random Forest regressor predicting monthly rent in INR. This educational prototype helps housing researchers and architects explore affordability from listing attributes. Underestimating rent can make a household budget unrealistic; overestimating it can exclude suitable options. MAE is the main metric because it expresses average error in INR/month; RMSE highlights costly large errors. It is not a pricing or lending decision system.

Earlier regression experiments and the furnishing-classification appendix are retained separately. The final published-model section reconstructs and evaluates the submitted pipeline.


In [ ]:
from pathlib import Path
ROOT=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'data/mumbai_cleaned.csv').exists())
FIG_DIR=ROOT/'figures';FIG_DIR.mkdir(exist_ok=True)
import matplotlib.pyplot as plt


In [ ]:
import pandas as pd
from pathlib import Path

data_path = ROOT/"data/mumbai_cleaned.csv"

df = pd.read_csv(data_path)

print("Dataset shape:", df.shape)
display(df.head())

In [ ]:
# Install pinned requirements before running the notebook.


In [ ]:
import sklearn

print("Scikit-learn version:", sklearn.__version__)

In [ ]:
import sys

print("Python executable:")
print(sys.executable)

print("\nPython version:")
print(sys.version)

print("\nChecking scikit-learn:")
%pip show scikit-learn


In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

print("All scikit-learn imports are working!")

Can we predict monthly rental prices in Mumbai based on property characteristics?

In [ ]:
print("Dataset shape:", df.shape)

print("\nAvailable columns:")
for col in df.columns:
    print(col)

print("\nColumn types:")
display(df.dtypes)

In [ ]:
print(df.columns.tolist())

## 3. Train/Test Split

We split the dataset into 80% training data and 20% test data.

The training set will be used for model training, cross-validation,
and hyperparameter tuning.

The test set will remain untouched until the final evaluation
to assess how well the model generalizes to unseen properties.

In [ ]:
from sklearn.model_selection import train_test_split

# Define the target and selected features
target = "Price_num"

features = [
    "Area_num",
    "BHK_num",
    "Bathrooms_num",
    "Balconies_num",
    "Furnished_clean",
    "Facing",
    "Location",
    "Property Type"
]

# Verify that all columns exist
assert all(col in df.columns for col in features + [target])

# Keep only rows with a valid target
model_df = df.dropna(subset=[target]).copy()

X = model_df[features]
y = model_df[target]

# Split the dataset: 80% training, 20% testing
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print("Training features:", X_train.shape)
print("Testing features:", X_test.shape)
print("Training target:", y_train.shape)
print("Testing target:", y_test.shape)

## 3. Preprocessing Pipeline

Numerical and categorical features require different preprocessing.

- Numerical features: median imputation and standardization.
- Categorical features: most-frequent imputation and one-hot encoding.

All preprocessing is fitted only on training data through a
scikit-learn pipeline to prevent data leakage.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# Numerical and categorical columns
numeric_features = [
    "Area_num",
    "BHK_num",
    "Bathrooms_num",
    "Balconies_num"
]

categorical_features = [
    "Furnished_clean",
    "Facing",
    "Location",
    "Property Type"
]

# Numerical preprocessing
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# Categorical preprocessing
categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

# Combine preprocessing steps
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

print("Preprocessing pipeline created successfully!")

## Underfitting and Overfitting Analysis
We investigated underfitting and overfitting using a Decision Tree Regressor to predict monthly rental prices in Mumbai. We varied the maximum tree depth to examine how model complexity affects prediction performance.

At a depth of 1, the model showed signs of underfitting, with high training and validation errors. As the depth increased, validation RMSE decreased, reaching its lowest value at a depth of 5.

Beyond a depth of 5, validation RMSE increased even though training RMSE continued to decrease. This indicates overfitting: the model learns the training data more closely but becomes less effective at predicting unseen properties.

Among the six tree depths tested, a maximum depth of 5 achieved the lowest validation RMSE. We therefore selected it as our current candidate for final evaluation on the held-out test set.

This experiment demonstrates that increasing model complexity does not always improve predictive performance. Selecting an appropriate level of complexity is important for building a model that generalises to unseen rental listings.

In [ ]:
# Install pinned requirements before running the notebook.


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, r2_score
import pandas as pd
import matplotlib.pyplot as plt

# Split the training data into a fitting set and a validation set
X_fit, X_val, y_fit, y_val = train_test_split(
    X_train,
    y_train,
    test_size=0.20,
    random_state=42
)

# Test different levels of model complexity
depths = [1, 2, 3, 5, 10, None]

results = []

for depth in depths:

    # Create a pipeline using our existing preprocessing
    model = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("regressor", DecisionTreeRegressor(
            max_depth=depth,
            random_state=42
        ))
    ])

    # Train the model
    model.fit(X_fit, y_fit)

    # Make predictions on training and validation data
    train_predictions = model.predict(X_fit)
    val_predictions = model.predict(X_val)

    # Calculate RMSE for both datasets
    train_rmse = mean_squared_error(
        y_fit, train_predictions
    ) ** 0.5

    val_rmse = mean_squared_error(
        y_val, val_predictions
    ) ** 0.5

    # Calculate R² scores
    train_r2 = r2_score(y_fit, train_predictions)
    val_r2 = r2_score(y_val, val_predictions)

    results.append({
        "Max Depth": str(depth),
        "Training RMSE": train_rmse,
        "Validation RMSE": val_rmse,
        "Training R²": train_r2,
        "Validation R²": val_r2
    })

# Display the results
results_df = pd.DataFrame(results)

display(results_df.round(2))


Plotting the results from the analysis of underfitting and overfitting.

In [ ]:
# Plot training and validation errors
plt.figure(figsize=(10, 5))

plt.plot(
    results_df["Max Depth"],
    results_df["Training RMSE"],
    marker="o",
    label="Training RMSE"
)

plt.plot(
    results_df["Max Depth"],
    results_df["Validation RMSE"],
    marker="o",
    label="Validation RMSE"
)

plt.title("Underfitting and Overfitting: Decision Tree")
plt.xlabel("Maximum Tree Depth")
plt.ylabel("RMSE (Rental Price Units)")
plt.legend()
plt.grid(True, alpha=0.3)

plt.savefig(FIG_DIR/'regression_underfitting_rmse.png',dpi=150,bbox_inches='tight')
plt.show()

## Comparison of Model Complexity

We compared three Decision Tree Regressor models with maximum depths of 1, 5, and unlimited depth to investigate underfitting and overfitting.

The model with a depth of 1 showed signs of underfitting, with a training RMSE of 103,880.85 and a validation RMSE of 93,969.27. This indicates that the model was too simple to capture the relationships between property characteristics and rental prices effectively.

The model with a depth of 5 achieved the lowest validation RMSE of 75,265.82. Its training RMSE was 76,347.14, indicating a relatively small difference between training and validation errors.

The unrestricted decision tree achieved a much lower training RMSE of 28,752.10, but its validation RMSE increased to 87,360.22. This indicates overfitting, as the model performed considerably better on the training data than on unseen validation data.

Based on these results, we selected a maximum tree depth of 5 as the best candidate among the three models compared. This experiment demonstrates the importance of controlling model complexity to improve generalisation.

In [ ]:
from sklearn.metrics import mean_absolute_error

# Select three model complexities for comparison
comparison_depths = [1, 5, None]

comparison_results = []

for depth in comparison_depths:

    model = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("regressor", DecisionTreeRegressor(
            max_depth=depth,
            random_state=42
        ))
    ])

    model.fit(X_fit, y_fit)

    train_predictions = model.predict(X_fit)
    val_predictions = model.predict(X_val)

    comparison_results.append({
        "Max Depth": str(depth),
        "Training MAE": mean_absolute_error(
            y_fit, train_predictions
        ),
        "Validation MAE": mean_absolute_error(
            y_val, val_predictions
        ),
        "Training RMSE": mean_squared_error(
            y_fit, train_predictions
        ) ** 0.5,
        "Validation RMSE": mean_squared_error(
            y_val, val_predictions
        ) ** 0.5
    })

comparison_df = pd.DataFrame(comparison_results)

display(comparison_df.round(2))

Maximum depth = 5 means we allow the decision tree to grow up to five levels deep, and in our experiment, this gave the lowest validation prediction error among the depths we tested.

In [ ]:
from sklearn.tree import DecisionTreeRegressor
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Select the best depth based on validation results
best_depth = 5

# Create the final model
final_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("regressor", DecisionTreeRegressor(
        max_depth=best_depth,
        random_state=42
    ))
])

# Train on the full original training set
final_model.fit(X_train, y_train)

# Predict rental prices for the unseen test set
test_predictions = final_model.predict(X_test)

# Calculate evaluation metrics
test_mae = mean_absolute_error(
    y_test, test_predictions
)

test_rmse = mean_squared_error(
    y_test, test_predictions
) ** 0.5

test_r2 = r2_score(
    y_test, test_predictions
)

# Display results
print("Final Model: Decision Tree Regressor")
print("Selected Maximum Depth:", best_depth)
print(f"Test MAE: {test_mae:.2f}")
print(f"Test RMSE: {test_rmse:.2f}")
print(f"Test R²: {test_r2:.3f}")

## Cross-Validation Analysis

We performed 5-fold cross-validation using a Decision Tree Regressor with a maximum depth of 5. The model achieved a mean validation RMSE of 79,945.09, a mean validation MAE of 39,698.33, and a mean validation R² score of 0.628.

The validation RMSE varied across the five folds, ranging from 65,970.28 to 105,663.98. The standard deviation of 16,066.60 indicates that predictive performance was not consistent across all subsets of the training data.

This variation suggests that the model may find some groups of rental properties more difficult to predict than others. Further investigation would be required to identify the reasons for these differences.

Overall, cross-validation provides a broader assessment of model performance than a single validation split. The results support further evaluation of the depth-5 decision tree, while highlighting the need to examine its prediction errors and compare it against a simple baseline model.

In [ ]:
from sklearn.model_selection import cross_validate
from sklearn.tree import DecisionTreeRegressor
from sklearn.pipeline import Pipeline
import pandas as pd
import numpy as np

# Create the decision tree model using the existing preprocessing
cv_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("regressor", DecisionTreeRegressor(
        max_depth=5,
        random_state=42
    ))
])

# Perform 5-fold cross-validation on the training data
cv_results = cross_validate(
    cv_model,
    X_train,
    y_train,
    cv=5,
    scoring={
        "rmse": "neg_root_mean_squared_error",
        "mae": "neg_mean_absolute_error",
        "r2": "r2"
    },
    return_train_score=True
)

# Convert negative error scores into positive values
cv_summary = pd.DataFrame({
    "Fold": range(1, 6),
    "Training RMSE": -cv_results["train_rmse"],
    "Validation RMSE": -cv_results["test_rmse"],
    "Training MAE": -cv_results["train_mae"],
    "Validation MAE": -cv_results["test_mae"],
    "Validation R²": cv_results["test_r2"]
})

display(cv_summary.round(2))

print("\nAverage cross-validation results:")
print(f"Mean Validation RMSE: {-cv_results['test_rmse'].mean():.2f}")
print(f"Mean Validation MAE: {-cv_results['test_mae'].mean():.2f}")
print(f"Mean Validation R²: {cv_results['test_r2'].mean():.3f}")

print("\nVariation in validation RMSE:")
print(f"Standard Deviation: {cv_results['test_rmse'].std():.2f}")

Plotting the folds in Cross-Validation.

In [ ]:
import matplotlib.pyplot as plt

# Number of folds
k = 5

fig, ax = plt.subplots(figsize=(10, 3.5))

# Draw the training and validation folds
for split in range(k):
    for fold in range(k):
        is_validation = fold == split

        ax.barh(
            y=split,
            width=1,
            left=fold,
            height=0.7,
            edgecolor="white",
            color="tab:orange" if is_validation else "tab:blue"
        )

    # Label each validation fold
    ax.text(
        k + 0.1,
        split,
        f"Validation: Fold {split + 1}",
        va="center",
        fontsize=9
    )

# Add the average score annotation
ax.text(
    0,
    k + 0.15,
    "Final cross-validation score = average of 5 validation scores",
    fontweight="bold",
    fontsize=10
)

# Configure axes
ax.set_yticks(range(k))
ax.set_yticklabels([f"Split {i + 1}" for i in range(k)])

ax.set_xticks([i + 0.5 for i in range(k)])
ax.set_xticklabels([f"Fold {i + 1}" for i in range(k)])

ax.invert_yaxis()
ax.set_xlim(0, k + 2.2)
ax.set_ylim(k + 0.5, -0.5)
ax.grid(False)

ax.set_title(
    "5-Fold Cross-Validation — Mumbai Rental Price Prediction",
    fontsize=12,
    pad=12
)

# Add legend
from matplotlib.patches import Patch

legend_elements = [
    Patch(facecolor="tab:blue", label="Training data"),
    Patch(facecolor="tab:orange", label="Validation data")
]

ax.legend(
    handles=legend_elements,
    loc="lower right",
    frameon=True
)

plt.tight_layout()
plt.savefig(FIG_DIR/'regression_cv_folds.png',dpi=150,bbox_inches='tight')
plt.show()

## Baseline Model Comparison

To evaluate whether using property characteristics improves rental-price prediction, we compared our Decision Tree Regressor with a baseline model that predicts the median rental price from the training data for every property.

The baseline achieved a test MAE of 67,123.75, a test RMSE of 140,522.95, and an R² score of -0.095.

In comparison, our Decision Tree with a maximum depth of 5 achieved a test MAE of 40,528.67, a test RMSE of 83,764.35, and an R² score of 0.611.

Compared with the baseline, the Decision Tree reduced MAE by approximately 39.6% and RMSE by approximately 40.4%. It also achieved a substantially higher R² score.

These results suggest that property characteristics provide useful information for predicting rental prices in our dataset. However, the remaining prediction errors indicate that the model could still be improved.

In [ ]:
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import pandas as pd

# Create a simple baseline model
baseline_model = DummyRegressor(strategy="median")

# Train the baseline using the original training data
baseline_model.fit(X_train, y_train)

# Predict prices for the original test data
baseline_predictions = baseline_model.predict(X_test)

# Calculate evaluation metrics
baseline_mae = mean_absolute_error(
    y_test, baseline_predictions
)

baseline_rmse = mean_squared_error(
    y_test, baseline_predictions
) ** 0.5

baseline_r2 = r2_score(
    y_test, baseline_predictions
)

# Display results
print("Baseline Model: Median Prediction")
print(f"Test MAE: {baseline_mae:.2f}")
print(f"Test RMSE: {baseline_rmse:.2f}")
print(f"Test R²: {baseline_r2:.3f}")

In [ ]:
# Compare the baseline and Decision Tree on the same test set
comparison_test = pd.DataFrame({
    "Model": [
        "Baseline (Median)",
        "Decision Tree (Depth 5)"
    ],
    "Test MAE": [
        baseline_mae,
        test_mae
    ],
    "Test RMSE": [
        baseline_rmse,
        test_rmse
    ],
    "Test R²": [
        baseline_r2,
        test_r2
    ]
})

display(comparison_test.round(3))

Inspecting Available Columns

In [ ]:
# Display all columns in the original dataset
print("Dataset columns:")
print(df.columns.tolist())

## Data Leakage Check

Our target variable is `Price_num`, which represents the rental price we want to predict.

We checked the eight features used by our Decision Tree Regressor and confirmed that the following columns are excluded from the model: `Price`, `Price_num`, `Price_per_m2`, and `Suspicious_Price`.

This is important because `Price_per_m2` may be calculated using the rental price, while `Suspicious_Price` may encode information derived from the target. Including such features could allow the model to access information that would not be available when predicting rent for a new property.

Our model instead uses property characteristics such as area, number of bedrooms, bathrooms, balconies, furnishing, facing, location, and property type.

This check confirms that the identified columns are not directly included in the model. Further checks of feature creation and preprocessing are still necessary to establish that the complete workflow is free from data leakage.

Investigating Potential Data Leakage

In [ ]:
import numpy as np
import pandas as pd

# Check the relationship between price, area, and price per square metre
leakage_check = df[
    ["Price_num", "Area_num", "Price_per_m2"]
].copy()

# Remove rows with missing or invalid values
leakage_check = leakage_check.dropna()

# Avoid division by zero
leakage_check = leakage_check[
    (leakage_check["Area_num"] > 0) &
    (leakage_check["Price_num"] > 0)
]

# Calculate price per square metre from the target and area
leakage_check["Calculated_Price_per_m2"] = (
    leakage_check["Price_num"] / leakage_check["Area_num"]
)

# Compare the calculated value with the existing feature
leakage_check["Difference"] = (
    leakage_check["Price_per_m2"] -
    leakage_check["Calculated_Price_per_m2"]
).abs()

print("Number of rows checked:", len(leakage_check))

print(
    "Mean absolute difference:",
    leakage_check["Difference"].mean()
)

print(
    "Maximum absolute difference:",
    leakage_check["Difference"].max()
)

display(leakage_check.head())

Compare model performance with and without the potential leakage feature

In [ ]:
from sklearn.base import clone
from sklearn.model_selection import cross_val_score
from sklearn.tree import DecisionTreeRegressor
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np
import pandas as pd

# Compare the existing features with a version excluding Price_per_m2
X_train_with = X_train.copy()
X_test_with = X_test.copy()

X_train_without = X_train.drop(
    columns=["Price_per_m2"],
    errors="ignore"
)

X_test_without = X_test.drop(
    columns=["Price_per_m2"],
    errors="ignore"
)

# Reuse the same preprocessing and model settings.
# Rebuild the preprocessing pipeline for each feature set if needed.
print("Training features with potential leakage:", X_train_with.shape[1])
print("Training features without Price_per_m2:", X_train_without.shape[1])

In [ ]:
print("Features used by our model:")
print(X_train.columns.tolist())

print("\nIs Price_per_m2 in X_train?")
print("Price_per_m2" in X_train.columns)

print("\nNumber of features:", len(X_train.columns))

In [ ]:
# Check the relationship between the target and our input features

print("Target variable:", y_train.name)

print("\nFeatures used for prediction:")
for column in X_train.columns:
    print("-", column)

print("\nChecking for target leakage:")

# Features that should not be used if they are derived from the target
potential_leakage_features = [
    "Price",
    "Price_num",
    "Price_per_m2",
    "Suspicious_Price"
]

for feature in potential_leakage_features:
    if feature in X_train.columns:
        print(f"WARNING: {feature} is included in the training features.")
    else:
        print(f"OK: {feature} is not included in the training features.")

Ensuring scikit learn is installed and working correctly to move forward to pipelines.

In [ ]:
import sklearn

print("Scikit-learn imported successfully!")
print("Version:", sklearn.__version__)

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor

print("All imports working correctly!")

Code cell to demonstrate the API using your existing rental dataset.

In [ ]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LinearRegression

# Select the numerical features
numeric_features = [
    "Area_num",
    "BHK_num",
    "Bathrooms_num",
    "Balconies_num"
]

# 1. Create preprocessing objects
imputer = SimpleImputer(strategy="median")
scaler = StandardScaler()

# 2. Learn the imputation values from training data
X_train_imputed = imputer.fit_transform(
    X_train[numeric_features]
)

# 3. Learn the scaling parameters and transform training data
X_train_scaled = scaler.fit_transform(X_train_imputed)

# 4. Train a regression model
model = LinearRegression()
model.fit(X_train_scaled, y_train)

print("Imputer, scaler, and model fitted successfully.")
print("Original numerical shape:", X_train[numeric_features].shape)
print("Transformed numerical shape:", X_train_scaled.shape)

Version 1: Preprocessing manually

In [ ]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

# Select numerical features
small_features = [
    "Area_num",
    "BHK_num",
    "Bathrooms_num",
    "Balconies_num"
]

# Create preprocessing tools and model
imputer = SimpleImputer(strategy="median")
scaler = StandardScaler()
model_manual = LinearRegression()

# 1. Fit and transform training data
X_train_step1 = imputer.fit_transform(
    X_train[small_features]
)

X_train_step2 = scaler.fit_transform(X_train_step1)

# 2. Train the model
model_manual.fit(X_train_step2, y_train)

# 3. Transform test data using learned parameters
X_test_step1 = imputer.transform(
    X_test[small_features]
)

X_test_step2 = scaler.transform(X_test_step1)

# 4. Generate predictions
predictions_manual = model_manual.predict(X_test_step2)

# 5. Evaluate
print(
    f"R² by hand: "
    f"{r2_score(y_test, predictions_manual):.4f}"
)

Version2: Using a Pipeline

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

# Create a pipeline
numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", LinearRegression())
])

# Fit the complete pipeline on training data
numeric_pipeline.fit(
    X_train[small_features],
    y_train
)

# Predict on test data
predictions_pipeline = numeric_pipeline.predict(
    X_test[small_features]
)

# Evaluate
print(
    f"R² with Pipeline: "
    f"{r2_score(y_test, predictions_pipeline):.4f}"
)

Compare the two approaches

In [ ]:
print(
    "Manual preprocessing R²:",
    round(r2_score(y_test, predictions_manual), 4)
)

print(
    "Pipeline R²:",
    round(r2_score(y_test, predictions_pipeline), 4)
)

A Pipeline reduces repetitive code, keeps preprocessing and model training together, and helps prevent data leakage by ensuring transformations are learned from training data. Our current pipeline uses only numerical features

Inspect the categories

In [ ]:
categorical_features = [
    "Furnished_clean",
    "Facing",
    "Location",
    "Property Type"
]

for column in categorical_features:
    print(f"\n--- {column} ---")
    print("Number of categories:", df[column].nunique(dropna=True))
    print(df[column].value_counts(dropna=False).head(10))

One-hot encoding

In [ ]:
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

# Select the Facing column from training data
facing_example = X_train[["Facing"]].head(10).copy()

# Step 1: Fill missing values
facing_imputer = SimpleImputer(strategy="most_frequent")
facing_filled = facing_imputer.fit_transform(facing_example)

# Step 2: Create the encoder with dense output
onehot = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=False
).set_output(transform="pandas")

# Step 3: Convert the filled data into a DataFrame
facing_filled_df = pd.DataFrame(
    facing_filled,
    columns=["Facing"],
    index=facing_example.index
)

# Step 4: Fit and transform the categorical data
facing_encoded = onehot.fit_transform(facing_filled_df)

# Display the result
display(facing_encoded)

Ordinal Encoding - Inspect furnishing categories

In [ ]:
import pandas as pd
from sklearn.preprocessing import OrdinalEncoder

print("Furnishing categories in training data:")
print(sorted(X_train["Furnished_clean"].dropna().unique()))

Apply ordinal encoding

In [ ]:
# Define the meaningful category order
furnishing_order = [
    "Unfurnished",
    "Semi-Furnished",
    "Furnished"
]

# Select non-missing training values for this demonstration
furnishing_example = X_train[["Furnished_clean"]].dropna()

# Create and fit the encoder
ordinal_encoder = OrdinalEncoder(
    categories=[furnishing_order]
)

encoded_furnishing = ordinal_encoder.fit_transform(
    furnishing_example
)

# Compare original and encoded values
furnishing_comparison = pd.DataFrame({
    "Original": furnishing_example["Furnished_clean"].values,
    "Encoded": encoded_furnishing.ravel()
})

display(furnishing_comparison.head(10))

Label Encoding - Facing type

In [ ]:
from sklearn.preprocessing import LabelEncoder

# Use non-missing training values for demonstration
facing_values = X_train["Facing"].dropna().astype(str)

# Create and fit the encoder
label_encoder = LabelEncoder()
facing_codes = label_encoder.fit_transform(facing_values)

# Display the category-to-code mapping
label_mapping = pd.DataFrame({
    "Category": label_encoder.classes_,
    "Code": range(len(label_encoder.classes_))
})

display(label_mapping)

Target Encoding - Location. First we check the version of sklearn if it supports.

In [ ]:
import sklearn

print("Scikit-learn version:", sklearn.__version__)

Encode locations - For the encoder to create one numerical feature representing the location's target-related information.

In [ ]:
from sklearn.preprocessing import TargetEncoder

location_encoder = TargetEncoder(
    target_type="continuous",
    random_state=42
)

# Fit and transform using training data only
location_encoded = location_encoder.fit_transform(
    X_train[["Location"]],
    y_train
)

print("Original shape:", X_train[["Location"]].shape)
print("Encoded shape:", location_encoded.shape)

print("\nFirst five encoded values:")
print(location_encoded[:5])

Column Transformer - Combining numerical preprocessing and categorical encoding.

In [ ]:
numeric_features = [
    "Area_num",
    "BHK_num",
    "Bathrooms_num",
    "Balconies_num"
]

nominal_features = [
    "Furnished_clean",
    "Facing",
    "Location",
    "Property Type"
]

Create the preprocessing pipelines

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# Numerical preprocessing
numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# Categorical preprocessing
nominal_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

# Combine the preprocessing steps
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_pipeline, numeric_features),
        ("nom", nominal_pipeline, nominal_features)
    ]
)

print("ColumnTransformer created successfully!")

Create the complete model Pipeline

In [ ]:
from sklearn.linear_model import LinearRegression

full_pipeline = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("model", LinearRegression())
])

print("Complete rental prediction pipeline created!")

Fit the Pipeline - For the Pipeline to learn the preprocessing parameters and trains the regression model using the training data.

In [ ]:
full_pipeline.fit(X_train, y_train)

print("Pipeline trained successfully!")

Inspect the transformed feature names

In [ ]:
feature_names = (
    full_pipeline
    .named_steps["preprocess"]
    .get_feature_names_out()
)

print("Original feature count:", X_train.shape[1])
print("Transformed feature count:", len(feature_names))

print("\nFirst 20 transformed feature names:")
print(feature_names[:20])

Display the Pipeline diagram

In [ ]:
from sklearn import set_config

set_config(display="diagram")

full_pipeline

Evaluate using cross-validation. 
To understand concepts of Consistent preprocessing.
Reduced risk of data leakage.
Easy model replacement.
Easier hyperparameter tuning.
Simpler cross-validation.

Evaluate using cross-validation

In [ ]:
from sklearn.model_selection import cross_validate

cv_results = cross_validate(
    full_pipeline,
    X_train,
    y_train,
    cv=5,
    scoring={
        "r2": "r2",
        "mae": "neg_mean_absolute_error",
        "rmse": "neg_root_mean_squared_error"
    },
    n_jobs=2
)

print("Five-fold cross-validation results")
print("----------------------------------")

print(
    "Mean R²:",
    round(cv_results["test_r2"].mean(), 4)
)

print(
    "Mean MAE:",
    round(-cv_results["test_mae"].mean(), 2)
)

print(
    "Mean RMSE:",
    round(-cv_results["test_rmse"].mean(), 2)
)

R² measures how well the model explains variation in rental prices.
MAE measures the average absolute prediction error.
RMSE penalizes larger prediction errors more heavily.

Replace Linear Regression with Random Forest

In [ ]:
from sklearn.ensemble import RandomForestRegressor

forest_pipeline = full_pipeline.set_params(
    model=RandomForestRegressor(
        n_estimators=100,
        random_state=42,
        n_jobs=1
    )
)

print("Estimator replaced with RandomForestRegressor.")

Inspect Pipeline parameters

In [ ]:
params = full_pipeline.get_params()

parameters_to_check = [
    "model",
    "preprocess__num__imputer__strategy",
    "preprocess__num__scaler__with_mean",
    "preprocess__nom__encoder__handle_unknown"
]

for parameter in parameters_to_check:
    print(f"{parameter}: {params[parameter]}")

Create the helper function: Reusable helper

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder


def make_pipeline_for(
    model,
    numeric,
    nominal,
    scale=True,
    nominal_encoder=None
):
    """
    Create a preprocessing and model pipeline.

    Parameters
    ----------
    model : scikit-learn estimator
        The regression model to train.

    numeric : list
        Numerical feature names.

    nominal : list
        Categorical feature names.

    scale : bool
        Whether to standardize numerical features.

    nominal_encoder : transformer or None
        Categorical encoder. Defaults to OneHotEncoder.

    Returns
    -------
    Pipeline
        A complete preprocessing and prediction pipeline.
    """

    # Numerical preprocessing
    numeric_steps = [
        ("imputer", SimpleImputer(strategy="median"))
    ]

    if scale:
        numeric_steps.append(
            ("scaler", StandardScaler())
        )

    numeric_pipeline = Pipeline(
        steps=numeric_steps
    )

    # Default categorical encoder
    if nominal_encoder is None:
        nominal_encoder = OneHotEncoder(
            handle_unknown="ignore"
        )

    # Categorical preprocessing
    nominal_pipeline = Pipeline(steps=[
        (
            "imputer",
            SimpleImputer(strategy="most_frequent")
        ),
        ("encoder", nominal_encoder)
    ])

    # Apply preprocessing to the relevant columns
    preprocessor = ColumnTransformer(
        transformers=[
            ("num", numeric_pipeline, numeric),
            ("nom", nominal_pipeline, nominal)
        ]
    )

    # Return the complete pipeline
    return Pipeline(steps=[
        ("preprocess", preprocessor),
        ("model", model)
    ])

Create a rental-specific helper

In [ ]:
def rental_pipeline(
    model,
    scale=True,
    nominal_encoder=None
):
    return make_pipeline_for(
        model=model,
        numeric=numeric_features,
        nominal=nominal_features,
        scale=scale,
        nominal_encoder=nominal_encoder
    )

Create a Linear Regression Pipeline

In [ ]:
from sklearn.linear_model import LinearRegression

rental_linear = rental_pipeline(
    model=LinearRegression(),
    scale=True
)

print("Linear regression pipeline created.")

Create a Random Forest Pipeline

In [ ]:
from sklearn.ensemble import RandomForestRegressor

rental_forest = rental_pipeline(
    model=RandomForestRegressor(
        n_estimators=100,
        random_state=42,
        n_jobs=2
    ),
    scale=False
)

print("Random forest pipeline created.")

Fitting both models tp have their own fitted preprocessing steps.

In [ ]:
rental_linear.fit(X_train, y_train)
print("Linear regression model trained.")

rental_forest.fit(X_train, y_train)
print("Random forest model trained.")

Saving and loading models: Create a model directory

In [ ]:
from pathlib import Path
import joblib

MODEL_DIR = ROOT/"models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

print("Model directory:", MODEL_DIR.resolve())

Saving the Linear Regression Pipeline [fitted model]

In [ ]:
model_path = MODEL_DIR / "mumbai_rental_pipeline.joblib"

joblib.dump(rental_linear, model_path)

print("Model saved successfully:", model_path)

Load the saved model

In [ ]:
loaded_model = joblib.load(model_path)

print("Saved model loaded successfully.")

In [ ]:
import numpy as np

# Select one row from the test dataset
sample_property = X_test.iloc[[0]]

# Predict using the original model
original_prediction = rental_linear.predict(
    sample_property
)

# Predict using the loaded model
loaded_prediction = loaded_model.predict(
    sample_property
)

print("Original prediction:", original_prediction[0])
print("Loaded prediction:", loaded_prediction[0])

# Confirm that the predictions match
assert np.allclose(
    original_prediction,
    loaded_prediction
)

print("Verification successful: predictions match!")

The Model Zoo — Types of Supervised Models

Understanding the types of supervised models. Since we want to predict rental prices, our task is a regression problem.

Import the required libraries

In [ ]:
import pandas as pd
import numpy as np

from sklearn.dummy import DummyRegressor
from sklearn.model_selection import cross_validate
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

Create and train the baseline model using existing rental_pipeline() function.

In [ ]:
# Create the baseline pipeline
baseline_pipeline = rental_pipeline(
    model=DummyRegressor(strategy="mean"),
    scale=False
)

# Train on the training data
baseline_pipeline.fit(X_train, y_train)

print("Baseline model trained successfully.")

Evaluating the baseline

In [ ]:
# Predict rental prices for the test data
baseline_predictions = baseline_pipeline.predict(X_test)

# Calculate evaluation metrics
baseline_r2 = r2_score(y_test, baseline_predictions)
baseline_mae = mean_absolute_error(y_test, baseline_predictions)
baseline_rmse = np.sqrt(
    mean_squared_error(y_test, baseline_predictions)
)

print("BASELINE MODEL PERFORMANCE")
print("--------------------------")
print(f"R²:   {baseline_r2:.4f}")
print(f"MAE:  {baseline_mae:.2f}")
print(f"RMSE: {baseline_rmse:.2f}")

### Inference: Baseline Model Performance

The baseline model achieved an **R² of approximately 0**, indicating that it performs no better than predicting the average rental price for every property.

- **MAE:** ₹77,182.53 — the average absolute prediction error.
- **RMSE:** ₹134,259.28 — the relatively high value indicates substantial prediction errors, particularly for some properties.

**Conclusion:** The baseline model has limited predictive power. More advanced regression models and feature engineering are needed to improve rental price predictions for the Mumbai housing market.

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
import matplotlib.pyplot as plt

# Define candidate models
models = {
    "Linear Regression": LinearRegression(),
    "KNN (k=5)": KNeighborsRegressor(n_neighbors=5),
    "Decision Tree (depth=3)": DecisionTreeRegressor(
        max_depth=3, random_state=42
    ),
    "Random Forest": RandomForestRegressor(
        n_estimators=200, random_state=42, n_jobs=2
    ),
    "Gradient Boosting": HistGradientBoostingRegressor(
        min_samples_leaf=3, random_state=42
    )
}

# Fit each model using the existing preprocessing pipeline
fig, axes = plt.subplots(1, 5, figsize=(20, 4))

for ax, (name, model) in zip(axes, models.items()):
    scale = name in ["Linear Regression", "KNN (k=5)"]

    pipeline = rental_pipeline(model=model, scale=scale)
    pipeline.fit(X_train, y_train)

    predictions = pipeline.predict(X_test)

    ax.scatter(y_test, predictions, s=15)
    ax.set_title(name)
    ax.set_xlabel("Actual rental price")
    ax.set_ylabel("Predicted rental price")

plt.tight_layout()
plt.savefig(FIG_DIR/'regression_five_models_predictions.png',dpi=150,bbox_inches='tight')
plt.show()

Inference: Comparison of Supervised Learning Models

The actual-versus-predicted plots compare five regression models for predicting Mumbai rental prices: Linear Regression, KNN, Decision Tree, Random Forest, and Gradient Boosting.

Linear Regression: Predictions generally follow the overall price pattern, but errors increase for expensive properties, suggesting difficulty capturing complex relationships.

KNN (k=5): Captures some variation in rental prices but shows noticeable prediction errors for high-priced properties and outliers.

Decision Tree (depth=3): Produces distinct horizontal prediction bands because of its limited depth. This indicates that the model is too simple to represent the full variation in rental prices.

Random Forest: Captures more complex patterns and produces a wider range of predictions, although substantial errors remain for some expensive properties.

Gradient Boosting: Captures nonlinear relationships and shows a prediction pattern similar to Random Forest, but some high-priced properties are still poorly estimated.

Comparing Supervised Models

In [ ]:
from sklearn.model_selection import cross_validate

results = []

for name, model in models.items():
    scale = name in ["Linear Regression", "KNN (k=5)"]
    pipe = rental_pipeline(model=model, scale=scale)

    scores = cross_validate(
        pipe, X_train, y_train, cv=5,
        scoring={
            "r2": "r2",
            "mae": "neg_mean_absolute_error",
            "rmse": "neg_root_mean_squared_error"
        },
        n_jobs=2
    )

    results.append({
        "Model": name,
        "R²": scores["test_r2"].mean(),
        "R2 std": scores["test_r2"].std(ddof=0),
            "MAE": -scores["test_mae"].mean(),
            "MAE std": scores["test_mae"].std(ddof=0),
        "RMSE": -scores["test_rmse"].mean(),
            "RMSE std": scores["test_rmse"].std(ddof=0)
    })

results_df = pd.DataFrame(results).sort_values("R²", ascending=False)
display(results_df.round(3))

### Inference: Model Comparison

Five-fold cross-validation was used to compare the regression models using R², Mean Absolute Error (MAE), and Root Mean Squared Error (RMSE).

- Higher R² indicates better explanatory performance.
- Lower MAE indicates smaller average prediction errors.
- Lower RMSE indicates fewer large prediction errors.

The results help identify promising models for predicting Mumbai rental prices. The model with the best cross-validation performance can be considered for further evaluation.

Visualize model performance

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

results_sorted = results_df.sort_values("R²")
axes[0].barh(results_sorted["Model"], results_sorted["R²"])
axes[0].set_title("Model Comparison: R²")
axes[0].set_xlabel("Mean CV R²")

results_sorted = results_df.sort_values("MAE")
axes[1].barh(results_sorted["Model"], results_sorted["MAE"])
axes[1].set_title("Model Comparison: MAE")
axes[1].set_xlabel("Mean CV MAE")

plt.tight_layout()
plt.savefig(FIG_DIR/'regression_model_comparison.png',dpi=150,bbox_inches='tight')
plt.show()

## Model Comparison: Inference

We compared five models using 5-fold cross-validation on the training set, looking at mean R² and mean MAE.

- R² ranking: Gradient Boosting (~0.64) > Linear Regression (~0.63) > Decision Tree, depth 3 (~0.62) > Random Forest (~0.61) > KNN, k=5 (~0.59). The whole spread is only about 0.05.
- MAE ranking: Random Forest (~37.5k) and Gradient Boosting (~38k) are lowest. KNN (~40k) is in the middle. Decision Tree (~42k) and Linear Regression (~42.5k) are highest.
- The two metrics disagreeLinear Regression is second on R² but worst on MAE, while Random Forest is fourth on R² but best on MAE. R² is driven by squared error, so it is dominated by a few very expensive listings. MAE reflects accuracy on a typical listing. The tree ensembles appear to predict typical rents better, while the linear model copes relatively better with extreme values.
- Gradient Boosting is the most balanced.** It is first on R² and second on MAE, so it is the strongest candidate overall.
- Differences are small.The gaps between models (~0.05 R², ~5k MAE) are smaller than the fold-to-fold variation seen earlier (RMSE std ≈ 16k), so we should not claim a decisive winner from this chart alone.
- Every model is far better than the baseline.** The median baseline had a test MAE of ~67k and R² < 0, so property characteristics clearly carry signal.

Selecting the best candidate model

In [ ]:
# Select the model with the highest mean cross-validation R²
best_model_name = results_df.iloc[0]["Model"]
best_model = models[best_model_name]

scale = best_model_name in ["Linear Regression", "KNN (k=5)"]

best_pipeline = rental_pipeline(
    model=best_model,
    scale=scale
)

best_pipeline.fit(X_train, y_train)

print("Selected model:", best_model_name)
print("Mean CV R²:", round(results_df.iloc[0]["R²"], 3))
print("Mean CV MAE:", round(results_df.iloc[0]["MAE"], 3))
print("Mean CV RMSE:", round(results_df.iloc[0]["RMSE"], 3))

Mumbai Rental Price Analysis

Regression Metrics

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

print("MAE measures average absolute error.")
print("RMSE penalizes large errors more heavily.")
print("R² measures how well the model explains price variation.")

### Inference: Regression Metrics

MAE, RMSE, and R² are used to evaluate rental price predictions. Lower MAE and RMSE indicate smaller prediction errors, while higher R² indicates better explanatory performance. 

Comparing models using cross-validation

In [ ]:
from sklearn.model_selection import cross_validate, KFold
import pandas as pd

cv = KFold(n_splits=5, shuffle=True, random_state=42)

def compare_models(models):
    results = []

    for name, model in models.items():
        scale = name in ["Linear Regression", "Ridge Regression", "KNN (k=5)"]
        pipe = rental_pipeline(model=model, scale=scale)

        scores = cross_validate(
            pipe, X_train, y_train, cv=cv,
            scoring={
                "r2": "r2",
                "mae": "neg_mean_absolute_error",
                "rmse": "neg_root_mean_squared_error"
            },
            n_jobs=2
        )

        results.append({
            "Model": name,
            "R²": scores["test_r2"].mean(),
            "R2 std": scores["test_r2"].std(ddof=0),
            "MAE": -scores["test_mae"].mean(),
            "MAE std": scores["test_mae"].std(ddof=0),
            "RMSE": -scores["test_rmse"].mean(),
            "RMSE std": scores["test_rmse"].std(ddof=0)
        })

    return pd.DataFrame(results).sort_values("R²", ascending=False)

### Inference: Cross-Validation

Five-fold cross-validation provides a more reliable comparison than a single validation split. The average scores help identify promising models while keeping the test dataset reserved for final evaluation.

In [ ]:
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder
from sklearn.linear_model import LinearRegression

encoding_results = []

encoders = {
    "One-Hot": OneHotEncoder(handle_unknown="ignore", sparse_output=False),
    "Ordinal": OrdinalEncoder(
        handle_unknown="use_encoded_value",
        unknown_value=-1
    )
}

for name, encoder in encoders.items():
    pipe = rental_pipeline(
        model=LinearRegression(),
        scale=True,
        nominal_encoder=encoder
    )

    scores = cross_validate(
        pipe, X_train, y_train, cv=cv,
        scoring="r2", n_jobs=2
    )

    encoding_results.append({
        "Encoding": name,
        "Mean CV R²": scores["test_score"].mean()
    })

display(pd.DataFrame(encoding_results).round(4))

### Inference: Categorical Encoding

The gap is small, but One-Hot is also the conceptually correct choice, since categories like `Location` and `Facing` have no natural order and Ordinal encoding imposes a false one. We therefore keep One-Hot encoding in the pipeline.

Comparing model families

In [ ]:
model_results = compare_models(models)
display(model_results.round(3))

### Inference: Model Families

Linear Regression (R² 0.620), Gradient Boosting (0.619) and Decision Tree (0.615) are effectively tied on R², so the data does not separate them clearly. Random Forest has the lowest MAE (about 37.8k) but a higher RMSE (about 83.7k), which suggests it predicts typical listings well but misses more on expensive ones. Linear Regression has the highest MAE (about 42.5k) but the lowest RMSE (about 80.5k), the reverse pattern. KNN is the weakest on R² and RMSE. Gradient Boosting offers the best balance, with near-top R² and the second-lowest MAE, so it remains the preferred candidate for tuning

Diagnose underfitting and overfitting

In [ ]:
from sklearn.model_selection import validation_curve
from sklearn.tree import DecisionTreeRegressor
import matplotlib.pyplot as plt

tree_pipe = rental_pipeline(
    model=DecisionTreeRegressor(random_state=42),
    scale=False
)

depths = [2, 4, 6, 10, 15, 20]

train_scores, val_scores = validation_curve(
    tree_pipe, X_train, y_train,
    param_name="model__max_depth",
    param_range=depths,
    cv=cv, scoring="r2", n_jobs=2
)

plt.plot(depths, train_scores.mean(axis=1), marker="o", label="Training")
plt.plot(depths, val_scores.mean(axis=1), marker="o", label="Validation")
plt.xlabel("Decision Tree Maximum Depth")
plt.ylabel("R²")
plt.title("Underfitting vs Overfitting")
plt.legend()
plt.savefig(FIG_DIR/'regression_validation_curve.png',dpi=150,bbox_inches='tight')
plt.show()

### Inference: Underfitting and Overfitting

Underfitting vs Overfitting (Decision Tree Depth)
-Depth 2: training and validation R² are both about 0.58, so the model is underfitting (too simple).
-Depth 4–6:validation R² peaks at about 0.63, the best bias–variance balance.
-Depth 10 and above:** training R² keeps climbing (up to about 0.95) while validation R² falls, a clear sign of overfitting.
-Validation R² drops to about 0.30 at depth 15. The model is memorising the training data.
-Conclusion: shallow tree (max depth around 4–6) generalises best, so depth should be constrained or tuned through cross-validation.

Hyperparameter tuning

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestRegressor

rf_pipe = rental_pipeline(
    model=RandomForestRegressor(random_state=42, n_jobs=1),
    scale=False
)

param_grid = {
    "model__n_estimators": [100, 200],
    "model__max_depth": [10, 20, None],
    "model__min_samples_leaf": [2, 5]
}

grid_search = GridSearchCV(
    rf_pipe,
    param_grid,
    cv=cv,
    scoring="r2",
    n_jobs=2
)

grid_search.fit(X_train, y_train)

print("Best parameters:", grid_search.best_params_)
print("Best CV R²:", round(grid_search.best_score_, 4))

best_pipeline = grid_search.best_estimator_

Final evaluation on the test set

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

test_predictions = best_pipeline.predict(X_test)

print("Final Test R²:", round(r2_score(y_test, test_predictions), 4))
print("Final Test MAE:", round(mean_absolute_error(y_test, test_predictions), 2))
print("Final Test RMSE:", round(np.sqrt(mean_squared_error(y_test, test_predictions)), 2))

### Inference: Final Test Evaluation

The model explains about 62% of the variance in rent on unseen data.RMSE is more than twice the MAE, which suggests a small number of high-rent properties produce very large errors. 

The rent distribution is right-skewed, and luxury listings are hard to predict.

Possible improvements: log-transform the target, cap or treat outliers, and add richer location features.

Assessing importance of features

In [ ]:
from sklearn.inspection import permutation_importance

importance = permutation_importance(
    best_pipeline, X_test, y_test,
    scoring="r2", n_repeats=5,
    random_state=42, n_jobs=2
)

importance_df = pd.DataFrame({
    "Feature": X_test.columns,
    "Importance": importance.importances_mean
}).sort_values("Importance", ascending=False)

display(importance_df)

importance_df.sort_values("Importance").plot(
    x="Feature", y="Importance",
    kind="barh", legend=False, figsize=(8, 5)
)
plt.title("Permutation Feature Importance")
plt.tight_layout()
plt.savefig(FIG_DIR/'regression_feature_importance.png',dpi=150,bbox_inches='tight')
plt.show()

### Inference: Feature Importance

-Area (sq ft) dominates. Shuffling it removes nearly all of the model's predictive power, so it is by far the strongest driver of rent.
-BHK and bathrooms contribute slightly, but they are strongly correlated with area.
-Facing, balconies and furnishing status have negligible influence.
-Location and property type show about zero importance. However, location is normally a key rent driver in Mumbai. Likely causes are high-cardinality encoding that wasn't effective, or the model simply preferring area. It is worth revisiting how location is encoded (e.g., target encoding or grouping localities into zones).

## Final Model Evaluation – Visual Analysis

We evaluate the selected Decision Tree Regressor (max_depth=5)
using the held-out test dataset.

Two visualizations are used:
1. Actual vs Predicted Rental Prices
2. Distribution of Prediction Errors

These plots help assess prediction accuracy, systematic bias,
and the model's ability to generalize to unseen properties.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Final selected model
final_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", DecisionTreeRegressor(max_depth=5, random_state=42))
])

final_model.fit(X_train, y_train)
y_pred = final_model.predict(X_test)

# Evaluation metrics
print(f"MAE:  {mean_absolute_error(y_test, y_pred):,.2f}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test, y_pred)):,.2f}")
print(f"R²:   {r2_score(y_test, y_pred):.3f}")

# Percentage prediction errors
actual = np.asarray(y_test)
errors = 100 * (y_pred - actual) / np.where(actual == 0, np.nan, actual)

# Create graphs
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Graph 1: Actual vs Predicted
axes[0].scatter(actual, y_pred, alpha=0.3, s=12)

limits = [min(actual.min(), y_pred.min()),
          max(actual.max(), y_pred.max())]

axes[0].plot(limits, limits, "r--", label="Perfect Prediction")
axes[0].set(
    xlabel="Actual Rent (₹)",
    ylabel="Predicted Rent (₹)",
    title="Actual vs Predicted Rental Prices"
)
axes[0].legend()
axes[0].grid(alpha=0.3)

# Graph 2: Prediction Error Distribution
axes[1].hist(errors[np.isfinite(errors)], bins=40,
             color="steelblue", edgecolor="white")
axes[1].axvline(0, color="red", linestyle="--")
axes[1].set(
    xlabel="Prediction Error (%)",
    ylabel="Number of Properties",
    title="Prediction Error Distribution"
)
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(FIG_DIR/'regression_predictions_errors.png',dpi=150,bbox_inches='tight')
plt.show()

# Prediction accuracy within ±20%
within_20 = np.nanmean(np.abs(errors) <= 20) * 100
print(f"{within_20:.1f}% of properties predicted within ±20% of actual rent")

Rental Price Error by Location — Horizontal Bar Chart

In [ ]:
location_errors = pd.DataFrame({
    "Location": X_test["Location"].values,
    "Absolute Error": np.abs(np.asarray(y_test) - y_pred)
})

location_mae = (
    location_errors.groupby("Location")["Absolute Error"]
    .agg(["mean", "count"])
    .query("count >= 5")
    .sort_values("mean", ascending=False)
    .head(10)
)

plt.figure(figsize=(10, 6))
plt.barh(location_mae.index, location_mae["mean"], color="coral")
plt.gca().invert_yaxis()

plt.xlabel("Mean Absolute Error (₹)")
plt.title("Top 10 Locations with Highest Prediction Errors")
plt.tight_layout()
plt.savefig(FIG_DIR/'regression_location_errors.png',dpi=150,bbox_inches='tight')
plt.show()

### Inference: Location-wise Prediction Errors

Prediction accuracy varies across Mumbai locations.Areas with higher MAE indicate greater difficulty in estimating
rental prices, potentially due to differences in property characteristics or limited training observations.

Saving and using the final model

In [ ]:
from pathlib import Path
import joblib

(ROOT/"models").mkdir(exist_ok=True)

joblib.dump(best_pipeline, ROOT/"models/mumbai_rental_model.joblib")

loaded_model = joblib.load(ROOT/"models/mumbai_rental_model.joblib")

# Example: predict for one property from the test dataset
sample_property = X_test.iloc[[0]]
predicted_price = loaded_model.predict(sample_property)[0]

print(f"Predicted rental price: {predicted_price:,.2f}")
print(f"Actual rental price:    {y_test.iloc[0]:,.2f}")

### Inference: Saving and Using the Model

The final pipeline is saved with `joblib` to `models/mumbai_rental_model.joblib`. Reloading it reproduces predictions without retraining.

Sample check: predicted rent is ₹25,586 against an actual ₹30,000, an error of about 15%. This is a reasonable single-point result, but one sample says little about overall accuracy; see the test metrics above.

## Appendix — furnishing classification (separate exploratory task)

These classification metrics do not describe the submitted rental-price model.
## A Quick Look at the Dataset

Our Mumbai property rental dataset contains *18,846 property listings and 26 columns*.

The Furnished_clean column contains three categories: Semi-Furnished (8,256 properties), Unfurnished (5,649 properties), and Furnished (4,928 properties). There are also 13 missing values.

For our classification task, we will investigate whether a machine-learning model can predict a property's furnishing category based on its other characteristics. Before training the model, we need to handle the missing target values and prepare our input features.

In [ ]:
print("Dataset shape:", df.shape)
print("\nFurnished status:")
print(df["Furnished_clean"].value_counts(dropna=False))

# Defining the Target Variable

We selected Furnished_clean as our target variable and removed the 13 property listings with missing furnishing categories. The resulting dataset contains 18,833 properties.

The target has three classes: Semi-Furnished (8,256), Unfurnished (5,649), and Furnished (4,928). Since the model must predict one of three categories, this is a *multiclass classification problem*.

We separated the target (y) from the input features (X). Before training, we must select appropriate features and exclude columns that could cause data leakage or would not be useful for predicting furnishing status.

In [ ]:
# Remove rows with missing target values
df_class = df.dropna(subset=["Furnished_clean"]).copy()

# Define the target and input features
target = "Furnished_clean"
X = df_class.drop(columns=[target])
y = df_class[target]

print("Dataset shape:", df_class.shape)
print("\nTarget distribution:")
print(y.value_counts())

In [ ]:
print(X.dtypes)

In [ ]:
features = [
    "Area_num", "BHK_num", "Bathrooms_num",
    "Balconies_num", "Facing", "Location", "Property Type"
]

X = df_class[features]
y = df_class["Furnished_clean"]

print("Features:", X.columns.tolist())
print("Target classes:", y.nunique())

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

## Building the Classification Pipeline

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier

numeric_features = ["Area_num", "BHK_num", "Bathrooms_num", "Balconies_num"]
categorical_features = ["Facing", "Location", "Property Type"]

preprocessor = ColumnTransformer([
    ("num", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]), numeric_features),
    ("cat", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore"))
    ]), categorical_features)
])

model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", DecisionTreeClassifier(random_state=42))
])

model.fit(X_train, y_train)
print("Classification model trained successfully!")

## Initial Model Evaluation

The Decision Tree Classifier achieved 42.8% test accuracy when predicting whether a property is furnished, semi-furnished, or unfurnished.

The model performs best on semi-furnished properties, with an F1-score of 0.52. Performance is weaker for furnished and unfurnished properties, with F1-scores of 0.34 and 0.36, respectively.

These results suggest that the model has limited predictive performance. We will investigate whether adjusting model complexity and using cross-validation can improve its generalisation.

In [ ]:
from sklearn.metrics import accuracy_score, classification_report

y_pred = model.predict(X_test)

print("Test Accuracy:", round(accuracy_score(y_test, y_pred), 3))
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

## Underfitting and Overfitting

As tree depth increases, training accuracy improves, but testing accuracy improves only slightly. At unlimited depth, training accuracy reaches 67.4%, while testing accuracy falls to 42.8%, indicating overfitting.

A maximum depth of 10 gives the highest testing accuracy (44.8%) among the tested options. However, overall performance remains limited, suggesting that tree depth alone does not solve the classification problem.

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

for depth in [1, 3, 5, 10, None]:
    tree = Pipeline([
        ("preprocessor", preprocessor),
        ("classifier", DecisionTreeClassifier(
            max_depth=depth, random_state=42
        ))
    ])

    tree.fit(X_train, y_train)

    train_acc = accuracy_score(y_train, tree.predict(X_train))
    test_acc = accuracy_score(y_test, tree.predict(X_test))

    print(f"Depth {depth}: Train accuracy = {train_acc:.3f}, Test accuracy = {test_acc:.3f}")

In [ ]:
# Historical recorded teaching scores, not recomputed by this plotting cell.
import matplotlib.pyplot as plt

depths = ["1", "3", "5", "10", "None"]
train_acc = [0.438, 0.447, 0.456, 0.490, 0.674]
test_acc = [0.438, 0.442, 0.444, 0.448, 0.428]

plt.figure(figsize=(8, 4))
plt.plot(depths, train_acc, marker="o", label="Training Accuracy")
plt.plot(depths, test_acc, marker="o", label="Testing Accuracy")

plt.xlabel("Maximum Tree Depth")
plt.ylabel("Accuracy")
plt.title("Historical recorded scores: Decision Tree")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(FIG_DIR/'classification_overfitting.png',dpi=150,bbox_inches='tight')
plt.show()

## 5-Fold Cross-Validation

The Decision Tree achieved a mean cross-validation accuracy of 41.6%, with a standard deviation of 0.004. The low variation indicates consistent performance across the five folds, although the overall accuracy remains limited.

This suggests that the model's performance is not strongly dependent on a particular training-validation split.

In [ ]:
from sklearn.model_selection import cross_val_score

cv_scores = cross_val_score(
    model, X_train, y_train,
    cv=5, scoring="accuracy"
)

print("Fold accuracies:", cv_scores.round(3))
print("Mean CV accuracy:", round(cv_scores.mean(), 3))
print("Standard deviation:", round(cv_scores.std(), 3))

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(7, 4))
plt.bar(range(1, 6), cv_scores)
plt.axhline(cv_scores.mean(), linestyle="--", label="Mean Accuracy")

plt.xlabel("Fold")
plt.ylabel("Accuracy")
plt.title("5-Fold Cross-Validation Accuracy")
plt.xticks(range(1, 6))
plt.legend()
plt.tight_layout()
plt.savefig(FIG_DIR/'classification_cv_accuracy.png',dpi=150,bbox_inches='tight')
plt.show()

The confusion matrix shows that the model performs best at identifying Semi-Furnished properties, correctly predicting 898 cases. However, it often misclassifies Furnished and Unfurnished properties as Semi-Furnished. This suggests that the model struggles to distinguish between furnishing categories. Overall, the results indicate that the model has limited classification performance.

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay
import matplotlib.pyplot as plt

ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred,
    cmap="Blues",
    xticks_rotation=45
)

plt.title("Confusion Matrix — Furnishing Classification")
plt.tight_layout()
plt.savefig(FIG_DIR/'classification_confusion_matrix.png',dpi=150,bbox_inches='tight')
plt.show()


The baseline model achieves 43.8% accuracy, while our Logistic Regression model achieves 42.8%. This shows that the model does not outperform the simple baseline. Accuracy alone can be misleading, so we also examine the classification report and confusion matrix to understand how well the model predicts each furnishing category.

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score

baseline = DummyClassifier(strategy="most_frequent")
baseline.fit(X_train, y_train)

baseline_pred = baseline.predict(X_test)

print("Baseline accuracy:", round(accuracy_score(y_test, baseline_pred), 3))
print("Model accuracy:", round(accuracy_score(y_test, y_pred), 3))

In [ ]:
print("Model:", type(model).__name__)
print("Probability predictions available:", hasattr(model, "predict_proba"))

The weighted multiclass ROC-AUC is 0.577, indicating limited ability to distinguish between Furnished, Semi-Furnished, and Unfurnished properties. Although the score is above 0.5, the model's performance remains weak. Together with the baseline accuracy of 43.8%, this suggests that the current features may not adequately capture the patterns needed for reliable classification.

In [ ]:
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import label_binarize

classes = model.classes_
y_test_binary = label_binarize(y_test, classes=classes)
y_prob = model.predict_proba(X_test)

auc = roc_auc_score(
    y_test_binary, y_prob,
    multi_class="ovr",
    average="weighted"
)

print("Weighted Multiclass ROC-AUC:", round(auc, 3))

In [ ]:
from sklearn.metrics import roc_curve, auc
from sklearn.preprocessing import label_binarize
import matplotlib.pyplot as plt

classes = model.classes_
y_test_binary = label_binarize(y_test, classes=classes)
y_prob = model.predict_proba(X_test)

for i, label in enumerate(classes):
    fpr, tpr, _ = roc_curve(y_test_binary[:, i], y_prob[:, i])
    plt.plot(fpr, tpr, label=f"{label} (AUC = {auc(fpr, tpr):.2f})")

plt.plot([0, 1], [0, 1], "k--", label="Random classifier")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("Multiclass ROC Curves — Furnishing Classification")
plt.legend()
plt.tight_layout()
plt.savefig(FIG_DIR/'classification_roc_curves.png',dpi=150,bbox_inches='tight')
plt.show()

The ROC curves show similar performance across all three furnishing categories, with AUC values between 0.57 and 0.59. Since these values are only slightly above 0.5, the model has limited ability to distinguish between classes. Unfurnished properties have the highest AUC (0.59), but the overall classification performance remains weak.

### Disguised Categorical Trap
The BHK, Bathrooms, and Balconies columns contain small sets of integer values. These represent meaningful property counts rather than arbitrary category codes, so treating them as numerical features is reasonable. The dataset also contains corresponding _num columns, which duplicate these features.

In [ ]:
# Check numeric columns with a small number of unique values
for col in df_class.select_dtypes(include="number").columns:
    if df_class[col].nunique() <= 10:
        print(col, ":", sorted(df_class[col].dropna().unique()))

The Decision Tree achieved an accuracy of 0.444, slightly outperforming the Dummy baseline at 0.438. However, the improvement is small, suggesting that the model has limited predictive power for classifying property furnishing types.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score

tree_model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", DecisionTreeClassifier(max_depth=5, random_state=42))
])

tree_model.fit(X_train, y_train)

dummy_model = DummyClassifier(strategy="most_frequent")
dummy_model.fit(X_train, y_train)

print("Decision Tree accuracy:",
      round(accuracy_score(y_test, tree_model.predict(X_test)), 3))

print("Dummy baseline accuracy:",
      round(accuracy_score(y_test, dummy_model.predict(X_test)), 3))

In [ ]:
from sklearn.tree import plot_tree
import matplotlib.pyplot as plt

tree = tree_model.named_steps["classifier"]
feature_names = tree_model.named_steps["preprocessor"].get_feature_names_out()

plt.figure(figsize=(20, 10))
plot_tree(
    tree,
    max_depth=3,
    feature_names=feature_names,
    class_names=tree.classes_,
    filled=True,
    rounded=True,
    fontsize=8
)
plt.savefig(FIG_DIR/'classification_decision_tree.png',dpi=150,bbox_inches='tight')
plt.show()

The decision tree uses property area as its primary feature for predicting furnishing type. It also considers facing direction and the number of bathrooms. Most nodes predict Semi-Furnished, suggesting that the model struggles to distinguish between the three furnishing categories. The tree provides insight into how the model makes predictions, but its accuracy remains limited.

In [ ]:
from sklearn.metrics import classification_report, accuracy_score

y_pred = tree_model.predict(X_test)

print("Final Test Accuracy:", round(accuracy_score(y_test, y_pred), 3))
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

The permutation importance analysis shows that Area_num is the most influential feature for predicting furnishing type, followed by Facing. Balconies_num has almost no impact on the model's macro F1 score, while Location, BHK_num, Property Type, and Bathrooms_num have importance values close to zero. These results suggest that the model relies mainly on property area, although its overall predictive performance remains limited.

In [ ]:
from sklearn.inspection import permutation_importance
from sklearn.metrics import f1_score
import pandas as pd

result = permutation_importance(
    tree_model,
    X_test,
    y_test,
    scoring="f1_macro",
    n_repeats=5,
    random_state=42
)

importance = pd.DataFrame({
    "Feature": X_test.columns,
    "Importance": result.importances_mean
}).sort_values("Importance", ascending=False)

print(importance)


In [ ]:
# Install pinned requirements before running the notebook.


# Which Parameters Matter

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(10, 6))

sns.barplot(
    data=importance.sort_values("Importance", ascending=False),
    x="Importance",
    y="Feature"
)

plt.title("Feature Importance for Predicting Furnishing Type")
plt.xlabel("Mean Decrease in Macro F1 Score")
plt.ylabel("Property Features")
plt.axvline(x=0, linestyle="--", linewidth=1)
plt.tight_layout()
plt.savefig(FIG_DIR/'classification_feature_importance.png',dpi=150,bbox_inches='tight')
plt.show()

### Summary: Which Model, Which Metric?

In this exercise, we used supervised machine learning to predict property furnishing type using features from the Mumbai rental listings dataset.

We trained and evaluated a Decision Tree classifier using features such as property area, number of bedrooms, bathrooms, balconies, facing direction, location, and property type.

**Key findings:**
- **Baseline accuracy:** 43.8%
- **Decision Tree test accuracy:** 44.4%
- **ROC-AUC:** Approximately 0.577
- **Most influential feature:** Property area, followed by facing direction.

The Decision Tree performed only slightly better than the baseline model. Its classification report also showed that it predicted Semi-Furnished properties much more successfully than Furnished or Unfurnished properties.

The feature-importance analysis suggested that property area and facing direction contributed most to predictions, while the other features had little measurable influence.

**Conclusion:** The model demonstrates how supervised learning can classify property listings, but its predictive performance is limited. Accuracy alone does not fully describe its performance, so metrics such as precision, recall, macro F1-score, and ROC-AUC are also useful.

In [ ]:
from sklearn.metrics import accuracy_score, classification_report

y_pred = tree_model.predict(X_test)

print("Final Test Accuracy:", round(accuracy_score(y_test, y_pred), 3))
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred))

## Final published model: Mumbai monthly rental regression

This section identifies the **single published model**: the tuned Random Forest regression pipeline, not the depth-5 Decision Tree or the furnishing classifier explored above. Earlier experiments and outputs are retained as the historical record. This section can run independently in a fresh kernel after installing the repository requirements.

The exported pipeline uses median numeric imputation **without scaling**, most-frequent categorical imputation, one-hot encoding with unknown categories ignored, and `RandomForestRegressor(n_estimators=200, max_depth=10, min_samples_leaf=5, random_state=42, n_jobs=1)`. These are the parameters selected by the earlier 12-configuration, five-fold grid search. We reuse the selected configuration; we do not retune on the test set.

The exact eight predictors are defined below. `Location` is constant (`Mumbai`), so this pipeline does not model neighborhoods. The source declares area in ft², but individual mixed-unit anomalies remain possible. The output is monthly INR rent, without a log transform. See [the Model Card](../models/README.md) for full assumptions, authors, MIT model/code licensing, separate CC BY 4.0 data attribution, and joblib security.

The official Hugging Face correction is [PR #4](https://huggingface.co/iaac-maai/group3-day2-mumbairental/discussions/4). Its reviewed artifact can be tested before merge with `predict_example.py --revision refs/pr/4`.


In [ ]:
from pathlib import Path
import hashlib
import json
import joblib
import numpy as np
import pandas as pd
import sklearn
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, r2_score, root_mean_squared_error
from sklearn.model_selection import KFold, cross_validate, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

# Find the repository from either its root or the notebook directory.
publication_root = next(
    parent for parent in [Path.cwd(), *Path.cwd().parents]
    if (parent / "data/mumbai_cleaned.csv").is_file()
)
publication_data_path = publication_root / "data/mumbai_cleaned.csv"
assert hashlib.sha256(publication_data_path.read_bytes()).hexdigest() == (
    "354d28e07432137cdc97a2197a556269137e74135af01186420e2e2917a19ee9"
), "Use the documented cleaned data snapshot"
assert sklearn.__version__ == "1.9.1", "Install the pinned requirements"
publication_features = [
    "Area_num", "BHK_num", "Bathrooms_num", "Balconies_num",
    "Furnished_clean", "Facing", "Location", "Property Type",
]
publication_data = pd.read_csv(publication_data_path).dropna(subset=["Price_num"])
publication_X_train, publication_X_test, publication_y_train, publication_y_test = train_test_split(
    publication_data[publication_features], publication_data["Price_num"],
    test_size=0.20, random_state=42,
)
publication_preprocessor = ColumnTransformer([
    ("num", Pipeline([("imputer", SimpleImputer(strategy="median"))]), publication_features[:4]),
    ("nom", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore")),
    ]), publication_features[4:]),
])
published_model = Pipeline([
    ("preprocess", publication_preprocessor),
    ("model", RandomForestRegressor(
        n_estimators=200, max_depth=10, min_samples_leaf=5,
        random_state=42, n_jobs=1,
    )),
])
published_model.fit(publication_X_train, publication_y_train)
publication_predictions = published_model.predict(publication_X_test)
publication_test = {
    "r2": r2_score(publication_y_test, publication_predictions),
    "mae": mean_absolute_error(publication_y_test, publication_predictions),
    "rmse": root_mean_squared_error(publication_y_test, publication_predictions),
}
np.testing.assert_allclose(
    [publication_test[key] for key in ["r2", "mae", "rmse"]],
    [0.6187457377846625, 38044.92790046044, 82899.45487397742],
    rtol=0, atol=1e-8,
)
print(f"Train rows: {len(publication_X_train)}; test rows: {len(publication_X_test)}")
print(json.dumps(publication_test, indent=2))


In [ ]:
publication_cv = KFold(n_splits=5, shuffle=True, random_state=42)
publication_scoring = {
    "r2": "r2", "mae": "neg_mean_absolute_error",
    "rmse": "neg_root_mean_squared_error",
}
publication_results = {}
for publication_name, publication_estimator in [
    ("Random Forest", clone(published_model)),
    ("Dummy mean", DummyRegressor(strategy="mean")),
    ("Dummy median", DummyRegressor(strategy="median")),
]:
    publication_fitted = clone(publication_estimator).fit(publication_X_train, publication_y_train)
    publication_scores = cross_validate(
        publication_estimator, publication_X_train, publication_y_train,
        cv=publication_cv, scoring=publication_scoring, n_jobs=1,
    )
    publication_result = {"test": {
        "r2": r2_score(publication_y_test, publication_fitted.predict(publication_X_test)),
        "mae": mean_absolute_error(publication_y_test, publication_fitted.predict(publication_X_test)),
        "rmse": root_mean_squared_error(publication_y_test, publication_fitted.predict(publication_X_test)),
    }, "cv": {}}
    for publication_metric in publication_scoring:
        publication_values = publication_scores["test_" + publication_metric] * (
            1 if publication_metric == "r2" else -1
        )
        publication_result["cv"][publication_metric] = {
            "folds": publication_values.tolist(),
            "mean": float(publication_values.mean()),
            "std": float(publication_values.std(ddof=0)),
        }
    publication_results[publication_name] = publication_result
print(json.dumps(publication_results, indent=2))


In [ ]:
# Export to the repository-level models directory; .gitignore keeps binaries out of Git.
publication_model_path = publication_root / "models/model.joblib"
publication_model_path.parent.mkdir(parents=True, exist_ok=True)

joblib.dump(published_model, publication_model_path)
# This file was just created locally from the pipeline above; never load unknown pickles.
publication_reloaded = joblib.load(publication_model_path)
np.testing.assert_allclose(
    publication_reloaded.predict(publication_X_test), publication_predictions,
    rtol=0, atol=1e-8,
)
publication_example = pd.DataFrame([{
    "Area_num": 330.0, "BHK_num": 1.0, "Bathrooms_num": 1.0,
    "Balconies_num": np.nan, "Furnished_clean": "Furnished", "Facing": "East",
    "Location": "Mumbai", "Property Type": "Apartment",
}])
publication_example_prediction = float(publication_reloaded.predict(publication_example)[0])
np.testing.assert_allclose(publication_example_prediction, 25586.094488364633, rtol=0, atol=1e-8)
print("Saved and reloaded: models/model.joblib")
print(f"Predicted monthly rent: INR {publication_example_prediction:,.2f}")
print("The new serialization may have a different file hash from the original Hub export.")
print("Use the original reviewed Hub artifact for the hash-pinned download example.")


### Interpretation and remaining limitations

The published forest has test R² **0.6187457377846625**, MAE **38044.92790046044 INR/month**, and RMSE **82899.45487397742 INR/month**. Five-fold training CV gives R² **0.6516348668185131 ± 0.07044628691519259**; standard deviation is across folds with `ddof=0`, not a confidence interval. The mean and median baselines above use exactly the same split and folds.

The CV folds were also used to tune the forest, so this is **non-nested CV** and is subject to selection optimism. Earlier experiments repeatedly inspected the same test set. There are 2,487 repeated predictor/target rows, and 752 test rows match training rows on all eight predictors plus target. These are dependence/leakage concerns, not proof of duplicate dwellings. A grouped split and fresh independent test set remain future validation work; no such experiment is claimed here.

`Location` is constant, so the earlier neighborhood explanations do not apply to this model. The depth-5 Decision Tree and classification figures above describe separate experiments; their scores must not be attached to this published forest. In particular, the reported 0.577 classification ROC-AUC belongs to the earlier unrestricted classifier, not the later depth-5 classifier. Original exploratory cells are retained for traceability.

The export cell regenerates the ignored local `models/model.joblib` on each run. Keep the reviewed Hub export separately; an equivalent freshly trained pipeline can serialize to different bytes. The published Hugging Face artifact is not changed by running this notebook.


## Published regression model — figures

The following figures use only the final published Random Forest predictions. Feature importance measures prediction sensitivity, not causality.


In [ ]:
fig,axes=plt.subplots(1,2,figsize=(11,4.5))
axes[0].scatter(publication_y_test,publication_predictions,s=8,alpha=.25)
limit=max(publication_y_test.max(),publication_predictions.max())
axes[0].plot([0,limit],[0,limit],color='black',ls='--')
axes[0].set(xlabel='Actual monthly rent (INR)',ylabel='Predicted monthly rent (INR)',title='Published forest: test predictions')
axes[1].scatter(publication_y_test,publication_predictions-publication_y_test,s=8,alpha=.25)
axes[1].axhline(0,color='black',ls='--')
axes[1].set(xlabel='Actual monthly rent (INR)',ylabel='Prediction minus actual (INR)',title='Published forest: residuals')
fig.tight_layout();fig.savefig(FIG_DIR/'published_predictions.png',dpi=160,bbox_inches='tight');plt.show()
from sklearn.inspection import permutation_importance
importance=permutation_importance(published_model,publication_X_test,publication_y_test,scoring='neg_mean_absolute_error',n_repeats=5,random_state=42,n_jobs=2)
order=np.argsort(importance.importances_mean)
fig,ax=plt.subplots(figsize=(8,4.5))
ax.barh(np.array(publication_features)[order],importance.importances_mean[order],xerr=importance.importances_std[order])
ax.set(xlabel='Increase in MAE after permutation (INR/month)',title='Published forest: permutation importance')
fig.tight_layout();fig.savefig(FIG_DIR/'published_importance.png',dpi=160,bbox_inches='tight');plt.show()
